# Модуль метода опорных мекторов

In [1]:
import pandas as pd

In [2]:
df_train = pd.read_csv("data\\train.csv", index_col='id')

In [3]:
X = df_train.iloc[:, :-1]
y = df_train.iloc[:, -1]

функция для кросс-валидации:

In [4]:
from sklearn.model_selection import cross_val_score

def cv(model, X, y):
    score = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='f1'
    )

    return score.mean() 

Применяем те же преобразования, что и в show_data.ipynb и исключим признаки, которые имеют много категориальных значений, добавим полиномиальные признаки, применим стандартизацию, потроим пайплайн обработки и оценим f1 score: 

In [5]:
from sklearn.svm import SVC
from sklearn.preprocessing import FunctionTransformer, PolynomialFeatures, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
import numpy as np


def preprocess(X):
    X = X.copy()

    X['job/study satisfaction'] = X['Job Satisfaction'].fillna(X['Study Satisfaction'])
    X['Work/Academic Pressure'] = X['Academic Pressure'].fillna(X['Work Pressure'])
    X.drop(columns=['Degree', 'Profession', 'Name', 'City', 'Job Satisfaction', 'Study Satisfaction', 'Academic Pressure', 'Work Pressure'], inplace=True)
    X.drop('CGPA', inplace=True, axis=1)

    return X

prep = FunctionTransformer(preprocess, validate=False)

num = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ('num', num, make_column_selector(dtype_include=np.number)),
    ("onehot", OneHotEncoder(sparse_output=False), ['Sleep Duration', 'Dietary Habits']),
    ("label", OrdinalEncoder(), ['Gender', 'Working Professional or Student', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']),
])

pipe = Pipeline([
    ("prep", prep),
    ("preprocess", preprocessor),
    ("model", SVC()),
])

cv(pipe, X, y)

np.float64(0.8665852110481171)

Сделаем подбор гиперпараметров:

In [6]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__C": [0.1, 1, 10],
    "model__kernel": ["linear", "rbf"],
    "model__gamma": ["scale", "auto"]
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1")

grid.fit(X, y)
grid.best_score_

np.float64(0.9659206930850509)

In [7]:
X_test = pd.read_csv("data\\test.csv", index_col='id') 

In [8]:
y_pred = grid.predict(X_test)
df_out = pd.DataFrame(y_pred, columns=["Depression"])
df_out.index = df_out.index + 1
df_out.to_csv("out\\svm.csv", index_label='id')

Попробуем дополнительно исключить признак Gender, откажемся от полиномальных признаков, вместо стандартизации используем нормализацию:

In [32]:
from sklearn.preprocessing import MinMaxScaler


def preprocess(X):
    X = X.copy()

    X['job/study satisfaction'] = X['Job Satisfaction'].fillna(X['Study Satisfaction'])
    X['Work/Academic Pressure'] = X['Academic Pressure'].fillna(X['Work Pressure'])
    X.drop(columns=['Gender', 'Degree', 'Profession', 'Name', 'City', 'Job Satisfaction', 'Study Satisfaction', 'Academic Pressure', 'Work Pressure'], inplace=True)
    X.drop('CGPA', inplace=True, axis=1)
    

    return X

prep = FunctionTransformer(preprocess, validate=False)

num = Pipeline([
    ("scaler", MinMaxScaler()),
])

preprocessor = ColumnTransformer([
    ('num', num, make_column_selector(dtype_include=np.number)),
    ("onehot", OneHotEncoder(sparse_output=False), ['Sleep Duration', 'Dietary Habits', 'Working Professional or Student', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']),
])

pipe = Pipeline([
    ("prep", prep),
    ("preprocess", preprocessor),
    ("model", SVC()),
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__kernel": ["rbf", "linear"],
    "model__gamma": ["scale", 0.01, 0.1, 1],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1")

grid.fit(X, y)
grid.best_score_

np.float64(0.9703037102071341)

In [33]:
y_pred = grid.predict(X_test)
df_out = pd.DataFrame(y_pred, columns=["Depression"])
df_out.index = df_out.index + 1
df_out.to_csv("out\\svm2.csv", index_label='id')

Изменим диапазон нормализации:

In [44]:
from sklearn.preprocessing import MinMaxScaler


def preprocess(X):
    X = X.copy()

    X['job/study satisfaction'] = X['Job Satisfaction'].fillna(X['Study Satisfaction'])
    X['Work/Academic Pressure'] = X['Academic Pressure'].fillna(X['Work Pressure'])
    X.drop(columns=['Gender', 'Degree', 'Profession', 'Name', 'City', 'Job Satisfaction', 'Study Satisfaction', 'Academic Pressure', 'Work Pressure'], inplace=True)
    X.drop('CGPA', inplace=True, axis=1)
    

    return X

prep = FunctionTransformer(preprocess, validate=False)

num = Pipeline([
    ("scaler", MinMaxScaler(feature_range=(-1, 1))),
])

preprocessor = ColumnTransformer([
    ('num', num, make_column_selector(dtype_include=np.number)),
    ("onehot", OneHotEncoder(sparse_output=False), ['Sleep Duration', 'Dietary Habits', 'Working Professional or Student', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']),
])

pipe = Pipeline([
    ("prep", prep),
    ("preprocess", preprocessor),
    ("model", SVC()),
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__kernel": ["rbf", "linear"],
    "model__gamma": ["scale", 0.01, 0.1, 1],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1")

grid.fit(X, y)
grid.best_score_

np.float64(0.9729772191673213)

In [45]:
y_pred = grid.predict(X_test)
df_out = pd.DataFrame(y_pred, columns=["Depression"])
df_out.index = df_out.index + 1
df_out.to_csv("out\\svm3.csv", index_label='id')